# Imports and Initialization

In [88]:
# imports
from google.cloud import compute_v1
import pandas as pd

In [89]:
# set variables
PROJECT_ID = "coms-amlc"

# accelerator client
accelerators_client = compute_v1.AcceleratorTypesClient()

# zones client
zones_client = compute_v1.ZonesClient()
zones = zones_client.list(project=PROJECT_ID)
test_zone = next(iter(zones), None)
print("test_zone = " + test_zone.name)

# regions client
regions_client = compute_v1.RegionsClient()
regions = regions_client.list(project=PROJECT_ID)
test_region = next(iter(regions), None)

print("test_region = " +test_region.name)

test_zone = us-east1-b
test_region = africa-south1


# Mapping Regions to Zones
This cell creates a mapping of regions to their zones and provides a function to find the region for a given zone name.

In [90]:
# Build region-to-zones map
region_zone_map = {}
zone_region_map = {}

for region in regions:
    region_name = region.name
    # region.zones is a list of full URLs, extract zone names
    zone_names = [z.split('/')[-1] for z in region.zones]
    region_zone_map[region_name] = zone_names
    for zone_name in zone_names:
        zone_region_map[zone_name] = region

def get_region_from_zone(zone_name):
    """Return the region for a given zone name."""
    return zone_region_map.get(zone_name, None)

# Example usage:
counter = 0
print("Region to Zones Map:")
for r, z in region_zone_map.items():
    print(f"{r}: {z}")
    counter = counter+1
    if counter > 5 : break

print("\nFind region for zone 'us-central1-a':", get_region_from_zone('us-central1-a').name)

Region to Zones Map:
africa-south1: ['africa-south1-b', 'africa-south1-a', 'africa-south1-c']
asia-east1: ['asia-east1-a', 'asia-east1-b', 'asia-east1-c']
asia-east2: ['asia-east2-c', 'asia-east2-b', 'asia-east2-a']
asia-northeast1: ['asia-northeast1-a', 'asia-northeast1-b', 'asia-northeast1-c']
asia-northeast2: ['asia-northeast2-b', 'asia-northeast2-c', 'asia-northeast2-a']
asia-northeast3: ['asia-northeast3-a', 'asia-northeast3-c', 'asia-northeast3-b']

Find region for zone 'us-central1-a': us-central1


# Mapping availability of GPUs with Regions and Zones

In [91]:
# Map GPU types to regions and zones
data = []

for zone in zones:
    try:
        gpus = accelerators_client.list(project=PROJECT_ID, zone=zone.name)
        for gpu in gpus:
            data.append({
                "GPU Type": gpu.name,
                "Region": get_region_from_zone(zone.name).name,
                "Zone": zone.name,
                "Available": "Yes"
            })
    except Exception as e:
        data.append({
            "GPU Type": "N/A",
            "Region": get_region_from_zone(zone.name).name,
            "Zone": zone.name
        })

In [92]:
# number of combinations
print(data.__sizeof__())

# Convert to DataFrame for easy viewing
gpu_df = pd.DataFrame(data)
gpu_df.head(20)  # Show first 20 rows

4200


,GPU Type,Region,Zone,Available
0,ct5lp,us-east1,us-east1-b,Yes
1,nvidia-b200,us-east1,us-east1-b,Yes
2,nvidia-l4,us-east1,us-east1-b,Yes
3,nvidia-l4-vws,us-east1,us-east1-b,Yes
4,nvidia-rtx-pro-6000,us-east1,us-east1-b,Yes
5,nvidia-rtx-pro-6000-vws,us-east1,us-east1-b,Yes
6,nvidia-tesla-a100,us-east1,us-east1-b,Yes
7,nvidia-tesla-p100,us-east1,us-east1-b,Yes
8,nvidia-tesla-p100-vws,us-east1,us-east1-b,Yes
9,ct5lp,us-east1,us-east1-c,Yes


In [93]:
# Extract unique GPU types from the data list
gpu_types = set(d["GPU Type"] for d in data if d["GPU Type"] != "N/A")

print("Available GPU types:")
for gpu_type in gpu_types:
    print(gpu_type)

Available GPU types:
nvidia-l4
nvidia-b200
nvidia-tesla-p4
nvidia-l4-vws
tpu7x
nvidia-rtx-pro-6000
nvidia-rtx-pro-6000-vws
nvidia-h100-mega-80gb
ct5l
nvidia-tesla-v100
nvidia-h100-80gb
nvidia-h200-141gb
ct5lp
nvidia-tesla-p100-vws
ct3
nvidia-tesla-t4-vws
nvidia-tesla-p100
ct6e
ct5p
nvidia-tesla-t4
ct3p
nvidia-a100-80gb
nvidia-tesla-p4-vws
nvidia-tesla-a100
nvidia-gb200


# Quota Availability

In [94]:
# Create a map of GPU quota metric name to its limit and usage
gpu_quota_map = {}

for region in regions:
    region_resource = regions_client.get(project=PROJECT_ID, region=region.name)
    for quota in region_resource.quotas:
        if "GPU" in quota.metric or "NVIDIA" in quota.metric:
            if(quota.limit > 0):
                gpu_quota_map[quota.metric] = {
                    "limit": quota.limit,
                    "usage": quota.usage
                }

In [95]:
# Print to check the map
{k: v for k, v in gpu_quota_map.items()}

{'NVIDIA_K80_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_P100_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_K80_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_P100_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_P100_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_V100_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_P4_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_P4_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_V100_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_P4_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_P100_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_P4_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_T4_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_T4_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_T4_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_T4_VWS_GPUS': {'limit': 1.0, 'usage': 0.0},
 'NVIDIA_L4_GPUS': {'limit': 1.0, 'usage': 0.0},
 'PREEMPTIBLE_NVIDIA_L4_GPUS': {'

In [96]:
import re
from collections import defaultdict

first = gpu_types
second = list(gpu_quota_map.keys())    # e.g. "NVIDIA_TESLA_P100_GPUS", "PREEMPTIBLE_NVIDIA_TESLA_T4_GPUS", etc.


# --- Normalize first set ---
def normalize_first(name):
    name = name.lower()
    name = name.replace("nvidia-tesla-", "")
    name = name.replace("nvidia-", "")
    return name  # e.g. "p100", "t4-vws", "l4"


# --- Normalize second set ---
def normalize_second(name):
    name = name.upper()
    # remove prefixes
    name = re.sub(r'^(PREEMPTIBLE_|COMMITTED_)', '', name)
    # remove suffix
    name = name.replace("_GPUS", "")
    # remove NVIDIA_
    name = name.replace("NVIDIA_", "")
    return name.lower()  # e.g. "p100", "p100_vws"


# --- Build lookup from first ---
first_lookup = defaultdict(list)
for f in first:
    key = normalize_first(f).replace("-", "_")  # align format
    first_lookup[key].append(f)


# --- Build mapping ---
gpu_naming_convention_matching = {}

for s in second:
    norm_s = normalize_second(s)
    # direct match
    if norm_s in first_lookup:
        gpu_naming_convention_matching[s] = first_lookup[norm_s]


# --- Pretty print ---
for k, v in gpu_naming_convention_matching.items():
    print(f"{k} -> {v}")

NVIDIA_P100_GPUS -> ['nvidia-tesla-p100']
PREEMPTIBLE_NVIDIA_P100_GPUS -> ['nvidia-tesla-p100']
NVIDIA_P100_VWS_GPUS -> ['nvidia-tesla-p100-vws']
NVIDIA_V100_GPUS -> ['nvidia-tesla-v100']
NVIDIA_P4_GPUS -> ['nvidia-tesla-p4']
NVIDIA_P4_VWS_GPUS -> ['nvidia-tesla-p4-vws']
PREEMPTIBLE_NVIDIA_V100_GPUS -> ['nvidia-tesla-v100']
PREEMPTIBLE_NVIDIA_P4_GPUS -> ['nvidia-tesla-p4']
PREEMPTIBLE_NVIDIA_P100_VWS_GPUS -> ['nvidia-tesla-p100-vws']
PREEMPTIBLE_NVIDIA_P4_VWS_GPUS -> ['nvidia-tesla-p4-vws']
NVIDIA_T4_GPUS -> ['nvidia-tesla-t4']
NVIDIA_T4_VWS_GPUS -> ['nvidia-tesla-t4-vws']
PREEMPTIBLE_NVIDIA_T4_GPUS -> ['nvidia-tesla-t4']
PREEMPTIBLE_NVIDIA_T4_VWS_GPUS -> ['nvidia-tesla-t4-vws']
NVIDIA_L4_GPUS -> ['nvidia-l4']
PREEMPTIBLE_NVIDIA_L4_GPUS -> ['nvidia-l4']
COMMITTED_NVIDIA_L4_GPUS -> ['nvidia-l4']


# Creating VM

In [98]:
def create_gpu_instance(
    project_id: str,
    zone: str,
    instance_name: str,
    machine_type: str = "n1-standard-4",
    gpu_type: str = "nvidia-tesla-t4",
    image_project: str = "deeplearning-platform-release",
    image_family: str = "pytorch-2-7-cu128-ubuntu-2404-nvidia-570"
) -> compute_v1.Operation:
    """
    Creates a Google Cloud Compute Engine VM instance with an attached GPU.
    Args:
        project_id: ID of the Google Cloud project.
        zone: The GCP zone to deploy the instance in (e.g., 'us-central1-a').
        instance_name: The name of the new VM instance.
        machine_type: The compute resources for the VM (e.g., 'n1-standard-4').
        gpu_type: The type of hardware accelerator to attach (e.g., 'nvidia-tesla-t4').
        image_project: The GCP project hosting the public boot image.
        image_family: The image family to use for the boot disk.
    Returns:
        compute_v1.Operation: An object representing the long-running VM creation operation.
    """
    
    # 1. Define instance basic properties
    instance = compute_v1.Instance(
        name=instance_name,
        machine_type=f"zones/{zone}/machineTypes/{machine_type}"
    )

    # 2. Configure boot disk
    initialize_params = compute_v1.AttachedDiskInitializeParams(
        source_image=f"projects/{image_project}/global/images/family/{image_family}"
    )
    boot_disk = compute_v1.AttachedDisk(
        boot=True,
        auto_delete=True,
        type_="PERSISTENT",
        initialize_params=initialize_params
    )
    instance.disks = [boot_disk]

    # 3. Configure network interface
    network_interface = compute_v1.NetworkInterface(
        name="global/networks/default"
    )
    instance.network_interfaces = [network_interface]

    # 4. Configure hardware accelerators (GPUs)
    accelerator = compute_v1.AcceleratorConfig(
        accelerator_type=f"zones/{zone}/acceleratorTypes/{gpu_type}", 
        accelerator_count=1
    )
    instance.guest_accelerators = [accelerator]

    # 5. Set metadata (Ensure Nvidia drivers are installed on startup)
    metadata = compute_v1.Metadata(
        items=[
            compute_v1.Items(key="install-nvidia-driver", value="true")
        ]
    )
    instance.metadata = metadata

    scheduling = compute_v1.Scheduling(
        on_host_maintenance="TERMINATE",
        automatic_restart=True
    )
    instance.scheduling = scheduling

    # 6. Submit the VM creation request
    instances_client = compute_v1.InstancesClient()
    operation = instances_client.insert(
        project=project_id,
        zone=zone,
        instance_resource=instance
    )
    return operation


In [104]:
ZONE = test_zone.name         
INSTANCE_NAME = "test-instance-2"

print(f"Initializing creation of VM: {INSTANCE_NAME} in zone: {ZONE}...")

GPU = list(gpu_naming_convention_matching.keys())[5]  
gpu_type = gpu_naming_convention_matching[GPU][0] if gpu_naming_convention_matching[GPU] else "nvidia-tesla-t4"  # default to t4 if no match
           
try:    
    # get zone availability for this GPU type
    filtered_data = [d for d in data if d["GPU Type"] == gpu_type]
    ZONE = filtered_data[0].get("Zone")
        
    operation_result = create_gpu_instance(
        project_id=PROJECT_ID,
        zone=ZONE,
        instance_name=INSTANCE_NAME,
        gpu_type=gpu_type
    )

    print("VM creation request successfully submitted.")
    print(f"Operation ID: {operation_result.name}")

except Exception as e:
    print(f"An error occurred while creating the VM: {e}")

Initializing creation of VM: test-instance-2 in zone: us-east1-b...
VM creation request successfully submitted.
Operation ID: operation-1772155642344-64bc4238c7492-45218e5a-d25998c4


In [105]:
instances_client = compute_v1.InstancesClient()
instance = instances_client.get(project=PROJECT_ID, zone=ZONE, instance=INSTANCE_NAME)
print(f"Name: {instance.name}")
print(f"Status: {instance.status}")
print(f"Machine Type: {instance.machine_type}")

Name: test-instance-2
Status: STOPPING
Machine Type: https://www.googleapis.com/compute/v1/projects/coms-amlc/zones/us-east4-c/machineTypes/n1-standard-4


In [102]:
print(operation_result.target_link)

https://www.googleapis.com/compute/v1/projects/coms-amlc/zones/us-east1-b/instances/test-instance-2
